Dans ce notebook on importe et merge différentes tables de données pour avoir un maximum de feature pouvant nous aider à prédire le prix au mètre carré d'un bien en Ile de France. Il y a un peu de traitement de données pour merge correctement les données. 

# Lib imports 

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
print(os.getcwd())
os.chdir("../")
print(os.getcwd())

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks
c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction


In [ ]:
from src.land_value_prediction.creation_base_de_donnees.conversion_semestre_trimestre_date import trimestre_to_date_fin
from src.land_value_prediction.creation_base_de_donnees.creation_lag import creer_lags
from src.land_value_prediction.creation_base_de_donnees.creation_poi import preparer_toutes_annees, traiter_dataset_complet
from land_value_prediction.creation_base_de_donnees.ajouter_prix_maisons_voisines import ajouter_prix_maisons_voisines_semestre_prec


# 1.Données géographiques, économiques et démographiques par code commune 

In [4]:
demographic_data = pd.read_excel("data/raw/data_par_commune/POPULATION_MUNICIPALE_COMMUNES_FRANCE.xlsx")
demographic_data.head()

,objectid,reg,dep,cv,codgeo,libgeo,p13_pop,p14_pop,p15_pop,p16_pop,p17_pop,p18_pop,p19_pop,p20_pop,p21_pop
0,115658,52,85,8502,85062,Châteauneuf,968.0,993.0,1013.0,1027.0,1056,1085.0,1114.0,1118.0,1134.0
1,115659,26,58,5808,58300,Urzy,1839.0,1835.0,1828.0,1802.0,1775,1749.0,1746.0,1747.0,1742.0
2,115660,43,70,7012,70137,Chassey-lès-Montbozon,218.0,217.0,216.0,215.0,217,215.0,215.0,220.0,225.0
3,115661,21,51,5123,51649,Vitry-le-François,13174.0,13144.0,12805.0,12552.0,12133,11743.0,11376.0,11458.0,11454.0
4,115662,11,78,7811,78638,Vaux-sur-Seine,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0


In [5]:
# reg = 11 c'est l'ile de France 
demographic_data = demographic_data[demographic_data["reg"] == 11]
demographic_data = demographic_data.rename(columns={"codgeo": "code_commune", 
                                                    'p13_pop': "pop_2016",
                                                    'p14_pop': "pop_2017",
                                                    'p15_pop': "pop_2018",
                                                    'p16_pop': "pop_2019",
                                                    'p17_pop': "pop_2020",
                                                    'p18_pop': "pop_2021",
                                                    'p19_pop': "pop_2022",
                                                    'p20_pop': "pop_2023",
                                                    'p21_pop': "pop_2024"})
demographic_data

,objectid,reg,dep,cv,code_commune,libgeo,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024
4,115662,11,78,7811,78638,Vaux-sur-Seine,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0
88,126398,11,77,7714,77295,Moisenay,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0
95,126405,11,95,9520,95301,Haute-Isle,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0
109,126419,11,78,7807,78402,Mézières-sur-Seine,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0
193,126588,11,91,9105,91578,Saint-Sulpice-de-Favières,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34911,110172,11,77,7716,77181,Ferrières-en-Brie,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0
34946,110207,11,95,9517,95446,Nesles-la-Vallée,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0
34952,110213,11,95,9509,95056,Belloy-en-France,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0
34966,110227,11,91,9103,91570,Saint-Michel-sur-Orge,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0


Ajout du taux de croissance annuelle de la population :

In [6]:
# Calculer les taux de croissance annuels pour toutes les années de 2019 à 2024
for annee in range(2019, 2025):
    nb_annees = annee - 2016
    demographic_data[f'taux_croissance_pop_annuel_2016_{annee}'] = (
        (demographic_data[f'pop_{annee}'] / demographic_data['pop_2016']) ** (1/nb_annees) - 1
    ) * 100

demographic_data = demographic_data.drop(columns=['reg', 'dep', 'cv', 'libgeo', 'objectid'])
demographic_data

,code_commune,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,taux_croissance_pop_annuel_2016_2019,taux_croissance_pop_annuel_2016_2020,taux_croissance_pop_annuel_2016_2021,taux_croissance_pop_annuel_2016_2022,taux_croissance_pop_annuel_2016_2023,taux_croissance_pop_annuel_2016_2024
4,78638,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214
88,77295,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217
95,95301,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286
109,78402,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379
193,91578,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34911,77181,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079
34946,95446,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766
34952,95056,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740
34966,91570,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221


In [7]:
demographic_data[demographic_data["code_commune"] == "75111"]

,code_commune,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,taux_croissance_pop_annuel_2016_2019,taux_croissance_pop_annuel_2016_2020,taux_croissance_pop_annuel_2016_2021,taux_croissance_pop_annuel_2016_2022,taux_croissance_pop_annuel_2016_2023,taux_croissance_pop_annuel_2016_2024
495,75111,153461.0,151542.0,149834.0,147017.0,146643,145903.0,145208.0,144292.0,142583.0,-1.419766,-1.129705,-1.005005,-0.91709,-0.876245,-0.914817


In [8]:
economic_data = pd.read_parquet("data/raw/data_par_commune/base-comparateur-de-territoires.parquet")
economic_data = economic_data.replace("s", pd.NA) # la valeur "s" signifie généralement que la donnée est confidentielle ou non disponible pour des raisons statistiques.
economic_data

,codgeo,nom_officiel_commune_arrondissement_municipal,p20_pop,p14_pop,superf,nais1420,dece1420,p20_men,naisd22,decesd22,...,etbe21,etfz21,etgu21,etgz21,etoq21,ettef121,ettefp1021,idf,geo_point,geo_shape
0,77121,Collégien,3339,3329,4.27,221,73,1258.000000,29,16,...,44,54,249,127,5,218,119,Oui,b'\x01\x01\x00\x00\x00\xe3\xf0U\xfa\xb6j\x05@\...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x004\x00\x0...
1,77249,Lésigny,7125,7387,10.13,331,243,2754.688016,53,34,...,3,21,108,26,19,127,13,Oui,b'\x01\x01\x00\x00\x00\x1eP\x9ct}\xed\x04@\xb7...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x81\x00...
2,77273,Marchémoret,594,557,7.04,24,8,202.907726,8,4,...,0,3,5,1,2,11,0,Oui,b'\x01\x01\x00\x00\x00W\xfb\xf2Y\x9c\x03\x06@G...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00N\x00\x0...
3,78010,Les Alluets-le-Roi,1215,1237,7.39,56,44,464.300630,12,6,...,1,15,37,17,5,50,9,Oui,b'\x01\x01\x00\x00\x000\xcb\x94\xff\xfe\xa3\xf...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00i\x00\x0...
4,78036,Autouillet,612,470,4.93,29,14,227.601970,7,1,...,1,0,2,0,2,5,0,Oui,b'\x01\x01\x00\x00\x00\xbe%\xdfc\x86\xc0\xfc?\...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00C\x00\x0...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,Verrières-le-Buisson,14602,15711,9.91,733,969,6091.947452,125,196,...,25,55,291,80,43,301,86,Oui,b'\x01\x01\x00\x00\x00\xd7\xb0YKO\x04\x02@j$\x...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x9e\x00...
1283,94079,Villiers-sur-Marne,29672,28278,4.33,2756,1080,12555.052456,552,194,...,22,125,418,107,59,492,76,Oui,b'\x01\x01\x00\x00\x00[\x95\xcfv\xbd\\\x04@\xf...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00O\x00\x0...
1284,95039,Auvers-sur-Oise,6792,6943,12.69,406,251,2876.344755,61,47,...,8,29,104,21,9,122,12,Oui,b'\x01\x01\x00\x00\x00z\xb9L\x7f\xc1<\x01@| (@...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00z\x00\x0...
1285,95042,Baillet-en-France,1893,2031,7.91,121,64,759.041633,20,10,...,10,19,47,17,5,62,16,Oui,b'\x01\x01\x00\x00\x00!\xe4\x17kBk\x02@\x86\xb...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00o\x00\x0...


In [9]:
economic_data = economic_data[['codgeo', 'superf', 'p20_men', 'p20_log', 'p20_rp', 'p20_rsecocc', 'p20_logvac',
       'p20_rp_prop', 'nbmenfisc20', 'med20', 'tp6020', 'p20_emplt',
       'p20_emplt_sal', 'p14_emplt', 'p20_pop1564', 'p20_chom1564',
       'p20_act1564', 'ettot21', 'etaz21', 'etbe21', 'etfz21', 'etgu21',
       'etgz21', 'etoq21', 'ettef121', 'ettefp1021']]

In [10]:
rename_cols = {
    'codgeo': 'code_commune',
    'superf': 'superficie_commune_km2',
    'p20_men': 'nb_menages_2020',
    'p20_log': 'nb_logements_total_2020',
    'p20_rp': 'nb_residences_principales_2020',
    'p20_rsecocc': 'nb_residences_secondaires_2020',
    'p20_logvac': 'nb_logements_vacants_2020',
    'p20_rp_prop': 'nb_residences_principales_proprietaires_2020',
    'med20': 'revenu_median_2020',
    'tp6020': 'taux_pauvrete_60pct_2020',
    'p20_emplt': 'emploi_total_2020',
    'p20_emplt_sal': 'emploi_salarie_2020',
    'p14_emplt': 'emploi_total_2014',
    'p20_pop1564': 'population_15_64_2020',
    'p20_chom1564': 'chomeurs_15_64_2020',
    'p20_act1564': 'nb_actifs_15_64_2020',
    'ettot21': 'etablissements_total_2021',
    'etaz21': 'etablissements_agriculture_2021',
    'etbe21': 'etablissements_industrie_2021',
    'etfz21': 'etablissements_construction_2021',
    'etgu21': 'etablissements_commerce_transport_2021',
    'etgz21': 'etablissements_services_entreprises_2021',
    'etoq21': 'etablissements_services_publics_sante_2021',
    'ettef121': 'etablissements_1_salarie_2021',
    'ettefp1021': 'etablissements_10_plus_salaries_2021',
    'nbmenfisc20': 'nb_menages_fiscaux_2020'
}

# Renommage
economic_data = economic_data.rename(columns=rename_cols)
economic_data

,code_commune,superficie_commune_km2,nb_menages_2020,nb_logements_total_2020,nb_residences_principales_2020,nb_residences_secondaires_2020,nb_logements_vacants_2020,nb_residences_principales_proprietaires_2020,nb_menages_fiscaux_2020,revenu_median_2020,...,nb_actifs_15_64_2020,etablissements_total_2021,etablissements_agriculture_2021,etablissements_industrie_2021,etablissements_construction_2021,etablissements_commerce_transport_2021,etablissements_services_entreprises_2021,etablissements_services_publics_sante_2021,etablissements_1_salarie_2021,etablissements_10_plus_salaries_2021
0,77121,4.27,1258.000000,1318.000000,1258.000000,18.000000,42.000000,845.000000,1242,25710,...,1694.000000,352,0,44,54,249,127,5,218,119
1,77249,10.13,2754.688016,2881.952676,2754.688016,24.050014,103.214645,2357.264137,2656,30460,...,3360.279613,153,2,3,21,108,26,19,127,13
2,77273,7.04,202.907726,215.562668,202.907726,2.920371,9.734570,167.480151,194,26550,...,342.333401,11,1,0,3,5,1,2,11,0
3,78010,7.39,464.300630,513.862381,464.300630,10.325365,39.236387,422.804167,478,35110,...,551.251291,63,5,1,15,37,17,5,50,9
4,78036,4.93,227.601970,264.167010,227.601970,24.028455,12.536585,209.978713,218,31960,...,309.157803,6,1,1,0,2,0,2,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,9.91,6091.947452,6565.061687,6091.947452,84.005691,389.108544,4393.282448,6227,35320,...,6363.848234,415,1,25,55,291,80,43,301,86
1283,94079,4.33,12555.052456,13288.446832,12555.052456,214.678221,518.716156,6385.956398,12878,23450,...,14828.293436,624,0,22,125,418,107,59,492,76
1284,95039,12.69,2876.344755,3138.832196,2876.344755,83.154419,179.333023,2236.154737,2853,28110,...,3347.074629,152,2,8,29,104,21,9,122,12
1285,95042,7.91,759.041633,799.142515,759.041633,4.010088,36.090794,648.178129,734,29430,...,991.564956,82,1,10,19,47,17,5,62,16


In [11]:
economic_data.isnull().mean() * 100

code_commune                                     0.000000
superficie_commune_km2                           0.000000
nb_menages_2020                                  0.000000
nb_logements_total_2020                          0.000000
nb_residences_principales_2020                   0.000000
nb_residences_secondaires_2020                   0.000000
nb_logements_vacants_2020                        0.000000
nb_residences_principales_proprietaires_2020     0.000000
nb_menages_fiscaux_2020                          1.243201
revenu_median_2020                               1.243201
taux_pauvrete_60pct_2020                        62.859363
emploi_total_2020                                0.000000
emploi_salarie_2020                              0.000000
emploi_total_2014                                0.000000
population_15_64_2020                            0.000000
chomeurs_15_64_2020                              0.000000
nb_actifs_15_64_2020                             0.000000
etablissements

On drop taux de pauvreté car trop de valeurs manquantes et cela sera surement capté par d'autres variables de revenus :

In [12]:
economic_data =economic_data.drop(columns=['taux_pauvrete_60pct_2020'])

Créer de nouveaux indicateurs :

In [13]:
# Densité de population (hab/km²)
economic_data["densite_population_active_2020"] = economic_data["population_15_64_2020"] / (economic_data["superficie_commune_km2"])

# Structure du logement
economic_data["taux_residences_secondaires"] = economic_data["nb_residences_secondaires_2020"] / economic_data["nb_logements_total_2020"]
economic_data["taux_logements_vacants"] = economic_data["nb_logements_vacants_2020"] / economic_data["nb_logements_total_2020"]
economic_data["taux_proprietaires"] = economic_data["nb_residences_principales_proprietaires_2020"] / economic_data["nb_residences_principales_2020"]
economic_data["ratio_residences_secondaires_population"] = economic_data["nb_residences_secondaires_2020"] / economic_data["population_15_64_2020"]
economic_data['densite_residences_principales_2020'] = (economic_data['nb_residences_principales_2020'] / (economic_data['superficie_commune_km2']))

# Emploi & chômage
economic_data["taux_chomage_15_64"] = economic_data["chomeurs_15_64_2020"] / economic_data["nb_actifs_15_64_2020"]
economic_data["taux_emploi"] = economic_data["emploi_total_2020"] / economic_data["population_15_64_2020"]
economic_data["evolution_emploi_2014_2020"] = (economic_data["emploi_total_2020"] - economic_data["emploi_total_2014"]) / economic_data["emploi_total_2014"]

# Activité économique
economic_data["part_services_entreprises"] = economic_data["etablissements_services_entreprises_2021"] / economic_data["etablissements_total_2021"]
economic_data["part_commerce_tourisme"] = economic_data["etablissements_commerce_transport_2021"] / economic_data["etablissements_total_2021"]
economic_data["part_admin_sante"] = economic_data["etablissements_services_publics_sante_2021"] / economic_data["etablissements_total_2021"]

# Tissu économique local
economic_data["nb_etablissements_par_menage"] = economic_data["etablissements_total_2021"] / economic_data["nb_menages_2020"]
economic_data["taux_etablissements_10_plus"] = economic_data["etablissements_10_plus_salaries_2021"] / economic_data["etablissements_total_2021"]

economic_data

,code_commune,superficie_commune_km2,nb_menages_2020,nb_logements_total_2020,nb_residences_principales_2020,nb_residences_secondaires_2020,nb_logements_vacants_2020,nb_residences_principales_proprietaires_2020,nb_menages_fiscaux_2020,revenu_median_2020,...,ratio_residences_secondaires_population,densite_residences_principales_2020,taux_chomage_15_64,taux_emploi,evolution_emploi_2014_2020,part_services_entreprises,part_commerce_tourisme,part_admin_sante,nb_etablissements_par_menage,taux_etablissements_10_plus
0,77121,4.27,1258.000000,1318.000000,1258.000000,18.000000,42.000000,845.000000,1242,25710,...,0.008291,294.613583,0.083825,1.485098,0.173768,0.360795,0.707386,0.014205,0.279809,0.338068
1,77249,10.13,2754.688016,2881.952676,2754.688016,24.050014,103.214645,2357.264137,2656,30460,...,0.005474,271.933664,0.076722,0.217478,0.022532,0.169935,0.705882,0.124183,0.055542,0.084967
2,77273,7.04,202.907726,215.562668,202.907726,2.920371,9.734570,167.480151,194,26550,...,0.007175,28.822120,0.058589,0.104832,0.138353,0.090909,0.454545,0.181818,0.054212,0.000000
3,78010,7.39,464.300630,513.862381,464.300630,10.325365,39.236387,422.804167,478,35110,...,0.013537,62.828231,0.075026,0.496032,0.111444,0.269841,0.587302,0.079365,0.135688,0.142857
4,78036,4.93,227.601970,264.167010,227.601970,24.028455,12.536585,209.978713,218,31960,...,0.059716,46.166728,0.060957,0.086112,-0.008080,0.000000,0.333333,0.333333,0.026362,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,9.91,6091.947452,6565.061687,6091.947452,84.005691,389.108544,4393.282448,6227,35320,...,0.009830,614.727291,0.085815,0.403862,-0.113901,0.192771,0.701205,0.103614,0.068123,0.207229
1283,94079,4.33,12555.052456,13288.446832,12555.052456,214.678221,518.716156,6385.956398,12878,23450,...,0.010882,2899.550221,0.090059,0.274468,0.036547,0.171474,0.669872,0.094551,0.049701,0.121795
1284,95039,12.69,2876.344755,3138.832196,2876.344755,83.154419,179.333023,2236.154737,2853,28110,...,0.019182,226.662313,0.080417,0.209048,-0.140841,0.138158,0.684211,0.059211,0.052845,0.078947
1285,95042,7.91,759.041633,799.142515,759.041633,4.010088,36.090794,648.178129,734,29430,...,0.003248,95.959751,0.070349,0.434082,-0.027186,0.207317,0.573171,0.060976,0.108031,0.195122


In [14]:
economic_data = economic_data.drop(columns=["emploi_total_2014",
                                            "emploi_total_2020",
                                            "chomeurs_15_64_2020",
                                            "nb_residences_secondaires_2020",
                                            "nb_logements_total_2020",
                                            "nb_logements_vacants_2020", 
                                            "nb_residences_principales_proprietaires_2020",
                                            "etablissements_services_entreprises_2021",
                                            "etablissements_commerce_transport_2021",
                                            "etablissements_services_publics_sante_2021",
                                            "etablissements_total_2021",
                                            "nb_menages_2020",
                                            "nb_residences_principales_2020", 
                                            "nb_menages_fiscaux_2020",
                                            "population_15_64_2020",
                                            "nb_actifs_15_64_2020", 
                                            "emploi_salarie_2020"
                                            ])

In [15]:
economic_data.columns

Index(['code_commune', 'superficie_commune_km2', 'revenu_median_2020',
       'etablissements_agriculture_2021', 'etablissements_industrie_2021',
       'etablissements_construction_2021', 'etablissements_1_salarie_2021',
       'etablissements_10_plus_salaries_2021',
       'densite_population_active_2020', 'taux_residences_secondaires',
       'taux_logements_vacants', 'taux_proprietaires',
       'ratio_residences_secondaires_population',
       'densite_residences_principales_2020', 'taux_chomage_15_64',
       'taux_emploi', 'evolution_emploi_2014_2020',
       'part_services_entreprises', 'part_commerce_tourisme',
       'part_admin_sante', 'nb_etablissements_par_menage',
       'taux_etablissements_10_plus'],
      dtype='object')

Données de score de developemment humain par commune :

In [16]:
idh_data = pd.read_parquet("data/raw/data_par_commune/indice-de-developpement-humain-idh2-des-communes-dile-de-france.parquet")
idh_data

,objectid,insee,annee,sante_plaf,educ_plaf,revenu_plaf,idh2,nomcom,pop
0,238,77227,2013,0.720523,0.345290,0.484803,0.516872,Hermé,646
1,345,77336,2013,0.683653,0.516738,0.595381,0.598591,Neufmoutiers-en-Brie,921
2,354,77345,2013,0.480528,0.427034,0.540652,0.482738,Orly-sur-Morin,676
3,359,77352,2013,0.648913,0.495740,0.604881,0.583178,Ozouer-le-Voulgis,1837
4,447,77443,2013,0.414469,0.465897,0.557855,0.479407,Sancy,379
...,...,...,...,...,...,...,...,...,...
3892,3165,78090,1999,0.388818,0.178893,0.537241,0.368318,Bouafle,2014
3893,3194,78171,1999,0.713465,0.356684,0.572949,0.547699,Condé-sur-Vesgre,1043
3894,3635,93027,1999,0.384332,0.083452,0.077246,0.181677,La Courneuve,35301
3895,3802,95308,1999,0.660175,0.344287,0.587841,0.530768,Hérouville,598


In [17]:
rename_dict = {
    'insee': 'code_commune',
    'sante_plaf': 'sante_score_2013_commune',
    'educ_plaf': 'education_score_2013_commune',
    'revenu_plaf': 'revenu_score_2013_commune',
    'idh2': 'idh2_2013_commune'
}
idh_data = idh_data.rename(columns=rename_dict)
idh_data = idh_data.drop(columns=['objectid', 'nomcom', 'pop'])
idh_data_2013 = idh_data[idh_data['annee'] == "2013"].drop(columns=['annee']).reset_index(drop=True)
idh_data_2013

,code_commune,sante_score_2013_commune,education_score_2013_commune,revenu_score_2013_commune,idh2_2013_commune
0,77227,0.720523,0.345290,0.484803,0.516872
1,77336,0.683653,0.516738,0.595381,0.598591
2,77345,0.480528,0.427034,0.540652,0.482738
3,77352,0.648913,0.495740,0.604881,0.583178
4,77443,0.414469,0.465897,0.557855,0.479407
...,...,...,...,...,...
1294,91044,0.774009,0.619009,0.662485,0.685168
1295,91411,0.750100,0.712545,0.758648,0.740431
1296,91587,0.616425,0.561824,0.616408,0.598219
1297,95051,0.744042,0.506516,0.626639,0.625732


Données sur la criminalité par commune :

In [18]:
crimes_data = pd.read_parquet("data/raw/data_par_commune/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet")
crimes_data = crimes_data.rename(columns={'CODGEO_2025': 'code_commune'})
crimes_data

,code_commune,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,ndiff,767,2016,348,2016,1.362069,0.959839
2,01001,2016,Violences sexuelles,Victime,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
3,01001,2016,Vols avec armes,Infraction,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4714195,97617,2024,Destructions et dégradations volontaires,Infraction,93.0,6.674322,diff,13934,2017,3872,2017,NaN,NaN
4714196,97617,2024,Usage de stupéfiants,Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,7.454546,0.874061
4714197,97617,2024,Usage de stupéfiants (AFD),Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,8.714286,0.903075
4714198,97617,2024,Trafic de stupéfiants,Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,3.357143,0.270960


In [19]:
# Calcul du total et du taux moyen par commune sur toutes les années
crime_rate_data = crimes_data.groupby('code_commune').agg({
    'nombre': 'sum',  # total des crimes
    'insee_pop': 'mean'  # population moyenne sur les années
}).reset_index()

crime_rate_data['taux_criminalite_moyen'] = crime_rate_data['nombre'] / crime_rate_data['insee_pop']

crime_rate_data = crime_rate_data[['code_commune', 'taux_criminalite_moyen']]
crime_rate_data

,code_commune,taux_criminalite_moyen
0,01001,0.000000
1,01002,0.000000
2,01004,0.443338
3,01005,0.095197
4,01006,0.000000
...,...,...
34915,97613,0.154073
34916,97614,0.222190
34917,97615,0.386471
34918,97616,0.233955


Dataframe final d'indicateurs communaux

In [20]:
for df in [demographic_data, economic_data, idh_data_2013, crime_rate_data]:
    df['code_commune'] = df['code_commune'].astype(str)
    
# Rassembler tous les dataframes en un seul
data_par_commune = demographic_data.merge(economic_data, on='code_commune', how='left') \
                           .merge(idh_data_2013, on='code_commune', how='left') \
                           .merge(crime_rate_data, on='code_commune', how='left')

data_par_commune

,code_commune,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,...,part_services_entreprises,part_commerce_tourisme,part_admin_sante,nb_etablissements_par_menage,taux_etablissements_10_plus,sante_score_2013_commune,education_score_2013_commune,revenu_score_2013_commune,idh2_2013_commune,taux_criminalite_moyen
0,78638,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0,...,0.159574,0.723404,0.085106,0.047612,0.063830,0.486201,0.558575,0.626984,0.557253,0.258535
1,77295,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0,...,0.086957,0.347826,0.217391,0.043502,0.130435,0.595452,0.458679,0.606847,0.553659,0.053883
2,95301,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0,...,0.500000,0.500000,0.500000,0.014925,0.000000,0.607826,0.530290,0.609074,0.582397,0.000000
3,78402,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0,...,0.210526,0.500000,0.144737,0.051287,0.144737,0.655672,0.472022,0.589140,0.572278,0.180551
4,91578,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0,...,0.100000,0.700000,0.200000,0.080645,0.100000,0.893288,0.600301,0.717455,0.737015,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0,...,0.231788,0.695364,0.052980,0.093617,0.410596,0.653765,0.587293,0.627911,0.622990,0.386791
1283,95446,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0,...,0.180000,0.620000,0.120000,0.064908,0.080000,0.674299,0.628949,0.678839,0.660696,0.157489
1284,95056,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0,...,0.245283,0.452830,0.113208,0.064744,0.132075,0.644696,0.482658,0.600748,0.576034,0.223398
1285,91570,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0,...,0.160000,0.597778,0.124444,0.050965,0.162222,0.663929,0.521276,0.503936,0.563047,0.384172


Densité de population par année :

In [21]:
for year in range(2016, 2025):
    pop_col = f'pop_{year}'
    density_col = f'densite_pop_{year}'
    # Calcul : population / superficie
    data_par_commune[density_col] = data_par_commune[pop_col] / (data_par_commune['superficie_commune_km2'])

In [22]:
data_par_commune[["pop_2016", "superficie_commune_km2", "densite_pop_2016"]]

,pop_2016,superficie_commune_km2,densite_pop_2016
0,4749.0,8.45,562.011834
1,1314.0,8.72,150.688073
2,301.0,2.57,117.120623
3,3626.0,10.42,347.984645
4,326.0,4.37,74.599542
...,...,...,...
1282,2793.0,6.75,413.777778
1283,1799.0,13.46,133.655275
1284,2115.0,9.49,222.866175
1285,20057.0,5.29,3791.493384


In [23]:
data_par_commune = data_par_commune.drop(columns=['pop_2016', 'pop_2017', 'pop_2018', 'pop_2019',
       'pop_2020', 'pop_2021', 'pop_2022', 'pop_2023', 'pop_2024', 'superficie_commune_km2'])

In [24]:
data_par_commune.isnull().mean() * 100

code_commune                               0.000000
taux_croissance_pop_annuel_2016_2019       0.000000
taux_croissance_pop_annuel_2016_2020       0.000000
taux_croissance_pop_annuel_2016_2021       0.000000
taux_croissance_pop_annuel_2016_2022       0.000000
taux_croissance_pop_annuel_2016_2023       0.000000
taux_croissance_pop_annuel_2016_2024       0.000000
revenu_median_2020                         1.243201
etablissements_agriculture_2021            0.000000
etablissements_industrie_2021              0.000000
etablissements_construction_2021           0.000000
etablissements_1_salarie_2021              0.000000
etablissements_10_plus_salaries_2021       0.000000
densite_population_active_2020             0.000000
taux_residences_secondaires                0.000000
taux_logements_vacants                     0.000000
taux_proprietaires                         0.000000
ratio_residences_secondaires_population    0.000000
densite_residences_principales_2020        0.000000
taux_chomage

Il y a des données manquantes qui pourraient être retrouvées car il y a des changements de code insee. 

Pour identifier facilement les données provenant de cette table on va renommber les variable avec le prefixe commune_

In [25]:
for col in data_par_commune.columns:
    if col != "code_commune":
        data_par_commune = data_par_commune.rename(columns={col : f"commune_{col}"})

data_par_commune

,code_commune,commune_taux_croissance_pop_annuel_2016_2019,commune_taux_croissance_pop_annuel_2016_2020,commune_taux_croissance_pop_annuel_2016_2021,commune_taux_croissance_pop_annuel_2016_2022,commune_taux_croissance_pop_annuel_2016_2023,commune_taux_croissance_pop_annuel_2016_2024,commune_revenu_median_2020,commune_etablissements_agriculture_2021,commune_etablissements_industrie_2021,...,commune_taux_criminalite_moyen,commune_densite_pop_2016,commune_densite_pop_2017,commune_densite_pop_2018,commune_densite_pop_2019,commune_densite_pop_2020,commune_densite_pop_2021,commune_densite_pop_2022,commune_densite_pop_2023,commune_densite_pop_2024
0,78638,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214,28210,0,3,...,0.258535,562.011834,557.988166,566.627219,574.792899,583.076923,583.313609,592.899408,594.082840,601.538462
1,77295,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217,28610,1,1,...,0.053883,150.688073,152.866972,155.045872,157.224771,158.256881,158.600917,159.059633,158.142202,157.224771
2,95301,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286,24820,0,0,...,0.000000,117.120623,112.840467,108.560311,108.560311,108.171206,109.727626,111.284047,112.840467,112.840467
3,78402,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379,26560,2,3,...,0.180551,347.984645,350.000000,348.944338,350.863724,352.783109,355.758157,353.454894,362.380038,367.178503
4,91578,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407,35220,1,0,...,0.000000,74.599542,75.057208,72.540046,69.794050,67.276888,65.446224,63.615561,61.327231,61.784897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079,28250,1,12,...,0.386791,413.777778,414.962963,446.222222,477.333333,508.592593,516.000000,558.222222,562.370370,569.037037
1283,95446,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766,29650,3,4,...,0.157489,133.655275,132.243685,133.803863,135.364042,136.924220,135.512630,133.952452,132.243685,133.952452
1284,95056,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740,27310,2,7,...,0.223398,222.866175,227.818757,228.134879,229.399368,230.663857,232.982086,232.139094,233.614331,234.773446
1285,91570,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221,22570,1,29,...,0.384172,3791.493384,3761.058601,3810.964083,3755.387524,3734.971645,3774.102079,3872.211720,4026.086957,4052.362949


# 2. Données des ventes et de leurs caractéristiques  

#### Import des données 

In [26]:
# Charger et concaténer
years = ['2020', '2021', '2022', '2023', '2024', '2025']
df_vf = pd.concat([
    pd.read_csv(f"data/raw/initial_data_files/DVF_{year}.csv", sep=",", low_memory=False)
    for year in years
], ignore_index=True)

# Filtrer Île-de-France
departements_idf = ['75', '77', '78', '91', '92', '93', '94', '95']
raw_idf_data = df_vf[df_vf['code_departement'].isin(departements_idf)].copy()

# libérer mémoire 
del df_vf

raw_idf_data

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
1569723,2020-621368,2020-07-01,1,Vente,246600.0,40.0,NaN,SEN DES LONGUES RAIES,0500,77670.0,...,Maison,105.0,4.0,S,sols,NaN,NaN,268.0,2.817040,48.383233
1569724,2020-621368,2020-07-01,1,Vente,246600.0,NaN,NaN,LES LONGUES RAIES,B021,77670.0,...,NaN,NaN,NaN,S,sols,NaN,NaN,178.0,2.816955,48.383203
1569725,2020-621368,2020-07-01,1,Vente,246600.0,NaN,NaN,LES LONGUES RAIES,B021,77670.0,...,NaN,NaN,NaN,S,sols,NaN,NaN,14.0,2.816974,48.383369
1569726,2020-621368,2020-07-01,1,Vente,246600.0,NaN,NaN,LES LONGUES RAIES,B021,77670.0,...,NaN,NaN,NaN,T,terres,NaN,NaN,48.0,2.817111,48.383239
1569727,2020-621369,2020-07-01,1,Vente,424000.0,5.0,NaN,RUE COROT,0123,77310.0,...,Maison,134.0,4.0,S,sols,NaN,NaN,500.0,2.558849,48.529151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20102734,2025-502880,2025-06-27,1,Vente,550000.0,84.0,NaN,RUE VERGNIAUD,9679,75013.0,...,Appartement,61.0,3.0,NaN,NaN,NaN,NaN,NaN,2.343663,48.823258
20102735,2025-502881,2025-06-27,1,Vente,550386.1,24.0,NaN,RUE DE PONTOISE,7621,75005.0,...,Appartement,47.0,2.0,NaN,NaN,NaN,NaN,NaN,2.351236,48.849138
20102736,2025-502881,2025-06-27,1,Vente,550386.1,24.0,NaN,RUE DE PONTOISE,7621,75005.0,...,Dépendance,NaN,0.0,NaN,NaN,NaN,NaN,NaN,2.351236,48.849138
20102737,2025-502882,2025-06-25,1,Vente,24417580.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.283008,48.840781


#### Suppression des colonnes inutiles 

In [27]:
# Supprimer colonnes inutiles (revoir plus tard si on peut en réutiliser certaines)
colonnes_a_supprimer = [
    'numero_disposition', 'adresse_suffixe', 
    'ancien_code_commune', 'ancien_nom_commune', 'ancien_id_parcelle',
    'numero_volume', 'code_nature_culture', 'code_nature_culture_speciale', 
    'id_mutation', 'lot1_numero', 'lot2_numero', 'lot3_numero',
    'lot4_numero', 'lot5_numero', 'lot1_surface_carrez',
    'lot2_surface_carrez', 'lot3_surface_carrez', 'lot4_surface_carrez', 
    'lot5_surface_carrez', 'nature_culture', 'nature_culture_speciale',
    ]

raw_idf_data.drop(columns=colonnes_a_supprimer, inplace=True)
raw_idf_data.head()

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,id_parcelle,nombre_lots,code_type_local,type_local,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude
1569723,2020-07-01,Vente,246600.0,40.0,SEN DES LONGUES RAIES,0500,77670.0,77419,Saint-Mammès,77,77419000AI0863,0,1.0,Maison,105.0,4.0,268.0,2.817040,48.383233
1569724,2020-07-01,Vente,246600.0,NaN,LES LONGUES RAIES,B021,77670.0,77419,Saint-Mammès,77,77419000AI0917,0,NaN,NaN,NaN,NaN,178.0,2.816955,48.383203
1569725,2020-07-01,Vente,246600.0,NaN,LES LONGUES RAIES,B021,77670.0,77419,Saint-Mammès,77,77419000AI1074,0,NaN,NaN,NaN,NaN,14.0,2.816974,48.383369
1569726,2020-07-01,Vente,246600.0,NaN,LES LONGUES RAIES,B021,77670.0,77419,Saint-Mammès,77,77419000AI1076,0,NaN,NaN,NaN,NaN,48.0,2.817111,48.383239
1569727,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,77040000AD0094,0,1.0,Maison,134.0,4.0,500.0,2.558849,48.529151


#### Créer un code cadastral comme combinaison du code commune et de l'identifiant de la parcelle

In [28]:
lettres_parcelle = raw_idf_data['id_parcelle'].str.extract(r'([A-Za-z]+)', expand=False)
raw_idf_data["code_cadastral"] = raw_idf_data["code_commune"] + "_" + lettres_parcelle
raw_idf_data["code_cadastral"]

1569723     77419_AI
1569724     77419_AI
1569725     77419_AI
1569726     77419_AI
1569727     77040_AD
              ...   
20102734    75113_DK
20102735    75105_AC
20102736    75105_AC
20102737    75115_ES
20102738    75115_ES
Name: code_cadastral, Length: 2393734, dtype: object

#### Convertir date et créer variables temporelles

In [29]:
raw_idf_data['date_mutation'] = pd.to_datetime(raw_idf_data['date_mutation'], format='%Y-%m-%d', errors='coerce')
raw_idf_data['annee'] = raw_idf_data['date_mutation'].dt.year
raw_idf_data['semestre'] = raw_idf_data['date_mutation'].dt.year.astype(str) + '-S' + np.where(raw_idf_data['date_mutation'].dt.quarter.gt(2), 2, 1).astype(str)
raw_idf_data['trimestre'] = raw_idf_data['date_mutation'].dt.year.astype(str) + '-T' + raw_idf_data['date_mutation'].dt.quarter.astype(str)

# Calculer le semestre précédent (S-1)
date_s_minus_1 = raw_idf_data['date_mutation'] - pd.DateOffset(months=6)
semestre_precedent = np.where(date_s_minus_1.dt.quarter.gt(2), 2, 1)
raw_idf_data['semestre_precedent'] = date_s_minus_1.dt.year.astype(str) + '-S' + semestre_precedent.astype(str)

# Calculer l'année précédente
raw_idf_data['annee_precedente'] = (raw_idf_data['date_mutation'] - pd.DateOffset(years=1)).dt.year

# Calculer le trimestre précédent
date_t_minus_1 = raw_idf_data['date_mutation'] - pd.DateOffset(months=3)
raw_idf_data['trimestre_precedent'] = date_t_minus_1.dt.year.astype(str) + 'T' + date_t_minus_1.dt.quarter.astype(str)

raw_idf_data[['date_mutation', "semestre", "trimestre", "semestre_precedent", "annee_precedente", "trimestre_precedent"]]

,date_mutation,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent
1569723,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569724,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569725,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569726,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569727,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
...,...,...,...,...,...,...
20102734,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102735,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102736,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102737,2025-06-25,2025-S1,2025-T2,2024-S2,2024,2025T1


#### 1er filtre sur le périmètre de modélisation (valeurs foncières, nombre de pièces du bien et surface du bien incohérentes):

In [30]:
numeric_cols = ['valeur_fonciere', 'surface_reelle_bati', 'surface_terrain',
                'nombre_pieces_principales']
for col in numeric_cols:
    if col in raw_idf_data.columns:
        if raw_idf_data[col].dtype == 'object':
            raw_idf_data[col] = raw_idf_data[col].str.replace(',', '.').astype(float)
        else:
            raw_idf_data[col] = pd.to_numeric(raw_idf_data[col], errors='coerce')

# Filtre pour enlever des valeurs incohérentes 
raw_idf_data = raw_idf_data[
    (raw_idf_data['valeur_fonciere'] > 1000) &
    (raw_idf_data['surface_reelle_bati'] > 9) &
    (raw_idf_data['surface_reelle_bati'] < 1000) &
    (raw_idf_data['nombre_pieces_principales'] >= 1) &
    (raw_idf_data['nombre_pieces_principales'] <= 15)
].copy()

#### Création de la variable cible et de variables qui en découlent, suppression des ventes de 2020

Créons la variable cible :

In [31]:
raw_idf_data['prix_m2'] = raw_idf_data['valeur_fonciere'] / raw_idf_data['surface_reelle_bati']

Prix médian des maisons environnantes au semestre précédant la vente, sous condition d'un nombre de ventes représentatif 

In [33]:
raw_idf_data = ajouter_prix_maisons_voisines_semestre_prec(raw_idf_data, 5)
raw_idf_data

Il y a 54005 codes cadastraux avec 5 transactions ou moins, pour un total de 102652 codes cadastraux.
Ces prix cadastraux ne seront pas utilisés, on utilisera plutôt le prix médian par code commune
Il y a 3073 codes communes avec 5 transactions ou moins, pour un total de 12194 codes communes.
Ces prix communaux ne seront pas utilisés, on utilisera plutôt le prix médian par code postal
Il y a 98 codes postaux avec 5 transactions ou moins, pour un total de 5255 codes postaux.
Ces prix postaux ne seront pas utilisés, on utilisera plutôt le prix médian par code département
Il y a 0 codes départements avec 5 transactions ou moins, pour un total de 80 codes départements.
Ces prix départementaux ne seront pas utilisés


,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,code_cadastral,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines
0,2020-07-01,Vente,246600.0,40.0,SEN DES LONGUES RAIES,0500,77670.0,77419,Saint-Mammès,77,...,77419_AI,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2348.571429,NaN,NaN
1,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
2,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
3,2020-07-01,Vente,192000.0,140.0,RUE DU VERT BUISSON,3090,77550.0,77296,Moissy-Cramayel,77,...,77296_C,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2206.896552,NaN,NaN
4,2020-07-02,Vente,160000.0,54.0,RUE DE LA MAIRIE,0040,77123.0,77471,Tousson,77,...,77471_A,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,1142.857143,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924492,2025-06-23,Vente,835000.0,167.0,RUE SAINT MAUR,8699,75011.0,75111,Paris 11e Arrondissement,75,...,75111_AC,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,10246.206349,12032.165128
924493,2025-06-25,Vente,375000.0,4.0,RUE FABRE D EGLANTINE,3479,75012.0,75112,Paris 12e Arrondissement,75,...,75112_CM,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8611.111111,7810.610968
924494,2025-06-25,Vente,1370000.0,2.0,AV PAUL DOUMER,7149,75016.0,75116,Paris 16e Arrondissement,75,...,75116_DS,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,11321.981013,15165.439084
924495,2025-06-27,Vente,550000.0,84.0,RUE VERGNIAUD,9679,75013.0,75113,Paris 13e Arrondissement,75,...,75113_DK,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8432.055749,11676.455777


In [34]:
raw_idf_data.isnull().mean() * 100

date_mutation                       0.000000
nature_mutation                     0.000000
valeur_fonciere                     0.000000
adresse_numero                      2.011580
adresse_nom_voie                    0.000325
adresse_code_voie                   0.000000
code_postal                         0.000974
code_commune                        0.000000
nom_commune                         0.000000
code_departement                    0.000000
id_parcelle                         0.000000
nombre_lots                         0.000000
code_type_local                     0.000000
type_local                          0.000000
surface_reelle_bati                 0.000000
nombre_pieces_principales           0.000000
surface_terrain                    65.340829
longitude                           1.548301
latitude                            1.548301
code_cadastral                      0.000000
annee                               0.000000
semestre                            0.000000
trimestre 

On a 12% des ventes qui n'ont pas de référence de prix médian de la commune. Ce sont toutes les ventes du 2ème semestre de 2020 car c'est le début de l'historique. On supprimera ces ventes par la suite. 

In [35]:
raw_idf_data_2020 = raw_idf_data[raw_idf_data["date_mutation"] < "2021-01-01"]
raw_idf_data = raw_idf_data[raw_idf_data["date_mutation"] >= "2021-01-01"]


raw_idf_data.isnull().mean() * 100

date_mutation                       0.000000
nature_mutation                     0.000000
valeur_fonciere                     0.000000
adresse_numero                      1.786411
adresse_nom_voie                    0.000371
adresse_code_voie                   0.000000
code_postal                         0.001112
code_commune                        0.000000
nom_commune                         0.000000
code_departement                    0.000000
id_parcelle                         0.000000
nombre_lots                         0.000000
code_type_local                     0.000000
type_local                          0.000000
surface_reelle_bati                 0.000000
nombre_pieces_principales           0.000000
surface_terrain                    65.666517
longitude                           1.417469
latitude                            1.417469
code_cadastral                      0.000000
annee                               0.000000
semestre                            0.000000
trimestre 

Calculer l'écart du prix du bien par rapport au prix médian de la commune observé au semestre précédent 


In [36]:
raw_idf_data['ecart_prix_median_pct'] = (
    (raw_idf_data['prix_m2'] - raw_idf_data['prix_m2_median_maisons_voisines']) /
    raw_idf_data['prix_m2_median_maisons_voisines'] * 100
)

raw_idf_data.head()

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
114885,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
114886,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
114887,2021-01-05,Vente,81000.0,2.0,RUE DES BASSES LOGES,0020,77210.0,77014,Avon,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1421.052632,3266.666667,3828.997115,-56.498389
114888,2021-01-08,Vente,165000.0,1.0,RUE DU MOULIN A VENT,1570,77127.0,77251,Lieusaint,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2500.000000,3749.833333,29169.298346,-33.330370
114889,2021-01-08,Vente,235000.0,8.0,RUE MIRABEAU,0550,77330.0,77350,Ozoir-la-Ferrière,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5731.707317,4077.741935,4254.384428,40.560815


#### 2ème filtre sur le périmètre: on se concentre sur les ventes et non le reste 

In [37]:
raw_idf_data['nature_mutation'].unique()

array(['Vente', "Vente en l'état futur d'achèvement", 'Adjudication',
       'Echange', 'Vente terrain à bâtir', 'Expropriation'], dtype=object)

On ne garde que les ventes, la plupart des ventes ne sont pas du neuf même si il peut y en avoir. On essaye de faire un modèle qui se concentre sur l'estimation de bien immobilier existant, donc on enlève "Vente en l'état futur d'achèvement" également. 

In [38]:
raw_idf_data = raw_idf_data[raw_idf_data['nature_mutation'].isin(['Vente'])].copy() 
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
114885,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
114886,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
114887,2021-01-05,Vente,81000.0,2.0,RUE DES BASSES LOGES,0020,77210.0,77014,Avon,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1421.052632,3266.666667,3828.997115,-56.498389
114888,2021-01-08,Vente,165000.0,1.0,RUE DU MOULIN A VENT,1570,77127.0,77251,Lieusaint,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2500.000000,3749.833333,29169.298346,-33.330370
114889,2021-01-08,Vente,235000.0,8.0,RUE MIRABEAU,0550,77330.0,77350,Ozoir-la-Ferrière,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5731.707317,4077.741935,4254.384428,40.560815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924492,2025-06-23,Vente,835000.0,167.0,RUE SAINT MAUR,8699,75011.0,75111,Paris 11e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,10246.206349,12032.165128,16.419395
924493,2025-06-25,Vente,375000.0,4.0,RUE FABRE D EGLANTINE,3479,75012.0,75112,Paris 12e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8611.111111,7810.610968,31.964809
924494,2025-06-25,Vente,1370000.0,2.0,AV PAUL DOUMER,7149,75016.0,75116,Paris 16e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,11321.981013,15165.439084,31.525608
924495,2025-06-27,Vente,550000.0,84.0,RUE VERGNIAUD,9679,75013.0,75113,Paris 13e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8432.055749,11676.455777,6.929955


#### 3ème filtre,  sur les valeurs incohérentes de la variable cible :

D'après se loger, le prix au metre carré en ile de France est compris entre :
- Prix bas : 1 662 €
- Prix haut : 14 879 €  

On va le confirmer par nos analyses :


In [39]:
mask_high_prices = raw_idf_data['prix_m2'] > 15000
suspicious_high_values_df = raw_idf_data[mask_high_prices]
suspicious_high_values_df

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
114976,2021-01-04,Vente,355000.0,14.0,RUE DE MELUN,0340,77220.0,77254,Liverdy-en-Brie,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,17750.000000,3000.000000,3148.526942,491.666667
115450,2021-01-13,Vente,5600000.0,28.0,RUE DE L INDUSTRIE,0630,77220.0,77470,Tournan-en-Brie,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,44800.000000,3488.906926,4242.867912,1184.069794
115451,2021-01-13,Vente,5600000.0,28.0,RUE DE L INDUSTRIE,0630,77220.0,77470,Tournan-en-Brie,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,44800.000000,3488.906926,4242.867912,1184.069794
115510,2021-01-13,Vente,565000.0,1.0,RUE GAMBETTA,0220,77170.0,77053,Brie-Comte-Robert,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,26904.761905,2989.130435,3180.282881,800.086580
115511,2021-01-13,Vente,565000.0,1.0,RUE GAMBETTA,0220,77170.0,77053,Brie-Comte-Robert,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,29736.842105,2989.130435,3180.282881,894.832536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924477,2025-06-30,Vente,250000.0,8.0,IMP SAINT SEBASTIEN,8754,75011.0,75111,Paris 11e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,16666.666667,11301.200686,16853.659279,47.476955
924478,2025-06-30,Vente,250000.0,8.0,IMP SAINT SEBASTIEN,8754,75011.0,75111,Paris 11e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,16666.666667,11301.200686,16853.659279,47.476955
924484,2025-06-26,Vente,1577000.0,10.0,RUE GUENEGAUD,4367,75006.0,75106,Paris 6e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,34282.608696,14875.000000,30325.361985,130.471319
924485,2025-06-26,Vente,1577000.0,10.0,RUE GUENEGAUD,4367,75006.0,75106,Paris 6e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,31540.000000,14875.000000,30325.361985,112.033613


In [40]:
suspicious_high_values_df["ecart_prix_median_pct"].describe()

count    9.732400e+04
mean     4.564138e+03
std      2.069458e+04
min     -9.974813e+01
25%      2.171114e+02
50%      9.193041e+02
75%      3.164114e+03
max      4.055456e+06
Name: ecart_prix_median_pct, dtype: float64

Les biens ayant un prix au mètre carré supérieurs à 15 000 euros sont en moyenne plus de 4000% plus hauts que le prix médian de leur commune. Complètement incohérent. 

In [41]:
mask_low_prices = raw_idf_data['prix_m2'] < 1500
suspicious_low_values_df = raw_idf_data[mask_low_prices]
suspicious_low_values_df

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
114887,2021-01-05,Vente,81000.0,2.0,RUE DES BASSES LOGES,0020,77210.0,77014,Avon,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1421.052632,3266.666667,3828.997115,-56.498389
114919,2021-01-06,Vente,99000.0,9001.0,AVE DE GAULLE 27 FOUGERES,A010,77210.0,77014,Avon,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1237.500000,3266.666667,3828.997115,-62.117347
114924,2021-01-05,Vente,60000.0,9001.0,RES DU PRIEURE,A020,77210.0,77014,Avon,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,967.741935,1944.047619,2087.215953,-50.220256
114954,2021-01-13,Vente,87500.0,641.0,RES DE LA BRETAGNE,A015,77190.0,77152,Dammarie-les-Lys,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1458.333333,1664.179104,1846.147985,-12.369208
114964,2021-01-12,Vente,102000.0,52.0,BD DE L ALMONT,0065,77000.0,77288,Melun,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,971.428571,1285.569620,1612.553111,-24.435942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924359,2025-06-26,Vente,18000.0,29.0,AV JEAN MOULIN,4928,75014.0,75114,Paris 14e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,418.604651,9426.764706,11365.834140,-95.559403
924360,2025-06-26,Vente,18000.0,29.0,AV JEAN MOULIN,4928,75014.0,75114,Paris 14e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,360.000000,9426.764706,11365.834140,-96.181086
924361,2025-06-26,Vente,18000.0,29.0,AV JEAN MOULIN,4928,75014.0,75114,Paris 14e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,360.000000,9426.764706,11365.834140,-96.181086
924400,2025-06-26,Vente,85000.0,39.0,AV VICTOR HUGO,9761,75016.0,75116,Paris 16e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,227.882038,12177.631579,13101.320508,-98.128683


In [42]:
suspicious_low_values_df["ecart_prix_median_pct"].describe()

count    19345.000000
mean       -69.589105
std         34.600517
min        -99.999837
25%        -95.510168
50%        -74.006849
75%        -53.835187
max       2452.380952
Name: ecart_prix_median_pct, dtype: float64

Les biens ayant un prix au mètre carré inférieur à 1 500 euros sont en moyenne 70% plus faibles que le prix médian de leur commune. Incohérent également. 

On va se séparer de ces données aberrantes. On aura peut être enlevé dans le paquet quelques ventes réelles à des prix extremes mais on preferera un modèle solide sur les transactions courantes que les ventes exceptionnelles. 

In [43]:
print(raw_idf_data.shape)
raw_idf_data = raw_idf_data[(~mask_high_prices) & (~mask_low_prices)]
print(raw_idf_data.shape)

(758895, 30)
(642226, 30)


On perd plus de 100 000 lignes. Ces ventes avaient potentiellement un impact sur le prix médian de la commune au semestre précédent. On va refaire les calculs pour recalculer un indicateurs plus représentatif. 

Commençons par appliquer les mêmes masques aux ventes de 2020 (utilisées pour calculer les prix medians de la commune au S-1)

In [44]:
raw_idf_data_2020.head(3)

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,code_cadastral,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines
0,2020-07-01,Vente,246600.0,40.0,SEN DES LONGUES RAIES,0500,77670.0,77419,Saint-Mammès,77,...,77419_AI,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2348.571429,NaN,NaN
1,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
2,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN


In [45]:
print(raw_idf_data_2020.shape)
raw_idf_data_2020 = raw_idf_data_2020[(raw_idf_data_2020["prix_m2"] < 15000) & (raw_idf_data_2020["prix_m2"] > 1500)]
print(raw_idf_data_2020.shape)

(114885, 29)
(94835, 29)


Ré-intégrons ces valeurs au dataframe :

In [46]:
raw_idf_data = pd.concat([raw_idf_data_2020, raw_idf_data], ignore_index=True)
raw_idf_data["date_mutation"].min()

Timestamp('2020-07-01 00:00:00')

Supprimons les colonnes précedemment créées :

In [47]:
raw_idf_data = raw_idf_data.drop(columns=["prix_m2_median_maisons_voisines", "prix_m2_moyen_maisons_voisines", "ecart_prix_median_pct"])
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,longitude,latitude,code_cadastral,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2
0,2020-07-01,Vente,246600.0,40.0,SEN DES LONGUES RAIES,0500,77670.0,77419,Saint-Mammès,77,...,2.817040,48.383233,77419_AI,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2348.571429
1,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,2.558849,48.529151,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104
2,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,2.558849,48.529151,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104
3,2020-07-01,Vente,192000.0,140.0,RUE DU VERT BUISSON,3090,77550.0,77296,Moissy-Cramayel,77,...,2.597956,48.627185,77296_C,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2206.896552
4,2020-07-02,Vente,175000.0,5086.0,BOISTRON,B013,77610.0,77104,Châtres,77,...,2.840918,48.720677,77104_YA,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,1988.636364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
737056,2025-06-23,Vente,835000.0,167.0,RUE SAINT MAUR,8699,75011.0,75111,Paris 11e Arrondissement,75,...,2.373088,48.869697,75111_AC,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429
737057,2025-06-25,Vente,375000.0,4.0,RUE FABRE D EGLANTINE,3479,75012.0,75112,Paris 12e Arrondissement,75,...,2.396254,48.846075,75112_CM,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364
737058,2025-06-25,Vente,1370000.0,2.0,AV PAUL DOUMER,7149,75016.0,75116,Paris 16e Arrondissement,75,...,2.284356,48.861611,75116_DS,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348
737059,2025-06-27,Vente,550000.0,84.0,RUE VERGNIAUD,9679,75013.0,75113,Paris 13e Arrondissement,75,...,2.343663,48.823258,75113_DK,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443


In [48]:
raw_idf_data = ajouter_prix_maisons_voisines_semestre_prec(raw_idf_data, 5)
raw_idf_data

Il y a 56000 codes cadastraux avec 5 transactions ou moins, pour un total de 100080 codes cadastraux.
Ces prix cadastraux ne seront pas utilisés, on utilisera plutôt le prix médian par code commune
Il y a 3288 codes communes avec 5 transactions ou moins, pour un total de 11992 codes communes.
Ces prix communaux ne seront pas utilisés, on utilisera plutôt le prix médian par code postal
Il y a 113 codes postaux avec 5 transactions ou moins, pour un total de 5249 codes postaux.
Ces prix postaux ne seront pas utilisés, on utilisera plutôt le prix médian par code département
Il y a 0 codes départements avec 5 transactions ou moins, pour un total de 80 codes départements.
Ces prix départementaux ne seront pas utilisés


,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,code_cadastral,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines
0,2020-07-01,Vente,246600.0,40.0,SEN DES LONGUES RAIES,0500,77670.0,77419,Saint-Mammès,77,...,77419_AI,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2348.571429,NaN,NaN
1,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
2,2020-07-01,Vente,424000.0,5.0,RUE COROT,0123,77310.0,77040,Boissise-le-Roi,77,...,77040_AD,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
3,2020-07-01,Vente,192000.0,140.0,RUE DU VERT BUISSON,3090,77550.0,77296,Moissy-Cramayel,77,...,77296_C,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2206.896552,NaN,NaN
4,2020-07-02,Vente,175000.0,5086.0,BOISTRON,B013,77610.0,77104,Châtres,77,...,77104_YA,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,1988.636364,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
737056,2025-06-23,Vente,835000.0,167.0,RUE SAINT MAUR,8699,75011.0,75111,Paris 11e Arrondissement,75,...,75111_AC,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,9934.451220,9766.551282
737057,2025-06-25,Vente,375000.0,4.0,RUE FABRE D EGLANTINE,3479,75012.0,75112,Paris 12e Arrondissement,75,...,75112_CM,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8972.222222,9107.834008
737058,2025-06-25,Vente,1370000.0,2.0,AV PAUL DOUMER,7149,75016.0,75116,Paris 16e Arrondissement,75,...,75116_DS,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,11010.101010,11348.667461
737059,2025-06-27,Vente,550000.0,84.0,RUE VERGNIAUD,9679,75013.0,75113,Paris 13e Arrondissement,75,...,75113_DK,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8571.428571,9411.259899


Supprimons à nouveau les valeurs de 2020 :

In [49]:
print(raw_idf_data.shape)
raw_idf_data = raw_idf_data[raw_idf_data["date_mutation"] >= "2021-01-01"]
print(raw_idf_data.shape)

(737061, 29)
(642226, 29)


Re-créons également la variable ecart_prix_median_pct

In [50]:
raw_idf_data['ecart_prix_median_pct'] = (
    (raw_idf_data['prix_m2'] - raw_idf_data['prix_m2_median_maisons_voisines']) /
    raw_idf_data['prix_m2_median_maisons_voisines'] * 100
)

raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
94835,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
94836,2021-01-05,Vente,352000.0,228.0,RUE DE L EGLISE,0220,77115.0,77453,Sivry-Courtry,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2661.494253,3048.879853,10.213777
94837,2021-01-08,Vente,165000.0,1.0,RUE DU MOULIN A VENT,1570,77127.0,77251,Lieusaint,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2500.000000,3238.636364,3449.092537,-22.807018
94838,2021-01-08,Vente,235000.0,8.0,RUE MIRABEAU,0550,77330.0,77350,Ozoir-la-Ferrière,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5731.707317,4077.741935,4254.384428,40.560815
94839,2021-01-06,Vente,334800.0,7.0,AV ROSCOMMON,0770,77590.0,77096,Chartrettes,77,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,3348.000000,3352.000000,3859.177752,-0.119332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
737056,2025-06-23,Vente,835000.0,167.0,RUE SAINT MAUR,8699,75011.0,75111,Paris 11e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,9934.451220,9766.551282,20.072777
737057,2025-06-25,Vente,375000.0,4.0,RUE FABRE D EGLANTINE,3479,75012.0,75112,Paris 12e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8972.222222,9107.834008,26.653532
737058,2025-06-25,Vente,1370000.0,2.0,AV PAUL DOUMER,7149,75016.0,75116,Paris 16e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,11010.101010,11348.667461,35.251296
737059,2025-06-27,Vente,550000.0,84.0,RUE VERGNIAUD,9679,75013.0,75113,Paris 13e Arrondissement,75,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8571.428571,9411.259899,5.191257


#### 4ème filtre sur le périmètre : ne garder que les observations pour lesquelles les coordonnées sont renseignées.

In [51]:
print(raw_idf_data.shape)
raw_idf_data = raw_idf_data[raw_idf_data[["latitude", "longitude"]].notnull().all(axis=1)].copy()
print(raw_idf_data.shape)
raw_idf_data[["latitude", "longitude"]].isnull().sum()


(642226, 30)
(634236, 30)


latitude     0
longitude    0
dtype: int64

Note : On pourrait retrouver ces coordonnées via une api du gouvernement mais cela échoue sur quelques adresses. Pour plus de faciliter on les enlève de la base. Voir le notebook "ajoute_long_let.ipynb" pour voir la façon dont retrouver les cooordonnées d'un bien. 

In [52]:
# Trier et réinitialiser l'index
raw_idf_data.sort_values('date_mutation', inplace=True)
raw_idf_data.reset_index(drop=True, inplace=True)

print(f"Dataset final de valeurs foncières : {raw_idf_data.shape}")

Dataset final de valeurs foncières : (634236, 30)


# 3. Merge entre la table des valeurs foncières et les indicateurs communaux

In [53]:
data_par_commune


,code_commune,commune_taux_croissance_pop_annuel_2016_2019,commune_taux_croissance_pop_annuel_2016_2020,commune_taux_croissance_pop_annuel_2016_2021,commune_taux_croissance_pop_annuel_2016_2022,commune_taux_croissance_pop_annuel_2016_2023,commune_taux_croissance_pop_annuel_2016_2024,commune_revenu_median_2020,commune_etablissements_agriculture_2021,commune_etablissements_industrie_2021,...,commune_taux_criminalite_moyen,commune_densite_pop_2016,commune_densite_pop_2017,commune_densite_pop_2018,commune_densite_pop_2019,commune_densite_pop_2020,commune_densite_pop_2021,commune_densite_pop_2022,commune_densite_pop_2023,commune_densite_pop_2024
0,78638,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214,28210,0,3,...,0.258535,562.011834,557.988166,566.627219,574.792899,583.076923,583.313609,592.899408,594.082840,601.538462
1,77295,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217,28610,1,1,...,0.053883,150.688073,152.866972,155.045872,157.224771,158.256881,158.600917,159.059633,158.142202,157.224771
2,95301,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286,24820,0,0,...,0.000000,117.120623,112.840467,108.560311,108.560311,108.171206,109.727626,111.284047,112.840467,112.840467
3,78402,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379,26560,2,3,...,0.180551,347.984645,350.000000,348.944338,350.863724,352.783109,355.758157,353.454894,362.380038,367.178503
4,91578,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407,35220,1,0,...,0.000000,74.599542,75.057208,72.540046,69.794050,67.276888,65.446224,63.615561,61.327231,61.784897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079,28250,1,12,...,0.386791,413.777778,414.962963,446.222222,477.333333,508.592593,516.000000,558.222222,562.370370,569.037037
1283,95446,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766,29650,3,4,...,0.157489,133.655275,132.243685,133.803863,135.364042,136.924220,135.512630,133.952452,132.243685,133.952452
1284,95056,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740,27310,2,7,...,0.223398,222.866175,227.818757,228.134879,229.399368,230.663857,232.982086,232.139094,233.614331,234.773446
1285,91570,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221,22570,1,29,...,0.384172,3791.493384,3761.058601,3810.964083,3755.387524,3734.971645,3774.102079,3872.211720,4026.086957,4052.362949


In [54]:
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,prix_m2_median_maisons_voisines,prix_m2_moyen_maisons_voisines,ecart_prix_median_pct
0,2021-01-01,Vente,169500.0,28.0,ALL HOCHE,4440,92130.0,92040,Issy-les-Moulineaux,92,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,9416.666667,8965.116279,8822.471633,5.036749
1,2021-01-02,Vente,185000.0,20.0,AV GAL LECLERC,0360,95250.0,95051,Beauchamp,95,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,4404.761905,3325.000000,3543.348626,32.474042
2,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5187.500000,4352.941176,5023.426740,19.172297
3,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5187.500000,4352.941176,5023.426740,19.172297
4,2021-01-04,Vente,255000.0,5.0,CHE DU MARCREUX,6115,93300.0,93001,Aubervilliers,93,...,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,3984.375000,4333.333333,4776.672150,-8.052885
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634231,2025-06-30,Vente,283000.0,4.0,RUE FRANCOIS MITTERRAND,0347,77380.0,77122,Combs-la-Ville,77,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,3723.684211,3621.912167,3636.210491,2.809898
634232,2025-06-30,Vente,144434.0,1.0,RUE JEAN DUSSART,1450,91390.0,91434,Morsang-sur-Orge,91,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,1569.934783,3579.545455,3485.129194,-56.141504
634233,2025-06-30,Vente,300000.0,32.0,RUE DU VAL ANDRE,0150,78560.0,78502,Le Port-Marly,78,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,5000.000000,4420.212766,4702.594848,13.116727
634234,2025-06-30,Vente,476000.0,1.0,IMP DE LA FORET,0322,78450.0,78674,Villepreux,78,...,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,3869.918699,4289.473684,4250.129567,-9.781036


In [55]:
data_par_commune.columns

Index(['code_commune', 'commune_taux_croissance_pop_annuel_2016_2019',
       'commune_taux_croissance_pop_annuel_2016_2020',
       'commune_taux_croissance_pop_annuel_2016_2021',
       'commune_taux_croissance_pop_annuel_2016_2022',
       'commune_taux_croissance_pop_annuel_2016_2023',
       'commune_taux_croissance_pop_annuel_2016_2024',
       'commune_revenu_median_2020', 'commune_etablissements_agriculture_2021',
       'commune_etablissements_industrie_2021',
       'commune_etablissements_construction_2021',
       'commune_etablissements_1_salarie_2021',
       'commune_etablissements_10_plus_salaries_2021',
       'commune_densite_population_active_2020',
       'commune_taux_residences_secondaires', 'commune_taux_logements_vacants',
       'commune_taux_proprietaires',
       'commune_ratio_residences_secondaires_population',
       'commune_densite_residences_principales_2020',
       'commune_taux_chomage_15_64', 'commune_taux_emploi',
       'commune_evolution_emploi_2

Pour éviter au maximum le leakage, on essaye d'utiliser quand on le peut les données de l'**année précédant la vente** :
- Vente en 2021 → données 2020
- Vente en 2024 → données 2023
- etc.

Règles :
1. **Variables avec millésime fixe** (2013, 2014, 2020, 2021) : toujours disponibles même si il y a un leak pour les ventes en 2020, 2021 qui utilisent des indicateurs de la même année. 
2. **Taux de croissance démographique** : on prend celui de l'année précédant la vente
3. **Densité de population** : on prend celle de l'année précédant la vente

In [56]:
colonnes_statiques = [
    'code_commune',
    'commune_revenu_median_2020',
    'commune_etablissements_agriculture_2021',
    'commune_etablissements_industrie_2021',
    'commune_etablissements_construction_2021',
    'commune_etablissements_1_salarie_2021',
    'commune_etablissements_10_plus_salaries_2021',
    'commune_densite_population_active_2020',
    'commune_taux_residences_secondaires',
    'commune_taux_logements_vacants',
    'commune_taux_proprietaires',
    'commune_ratio_residences_secondaires_population',
    'commune_densite_residences_principales_2020',
    'commune_taux_chomage_15_64',
    'commune_taux_emploi',
    'commune_evolution_emploi_2014_2020',
    'commune_part_services_entreprises',
    'commune_part_commerce_tourisme',
    'commune_part_admin_sante',
    'commune_nb_etablissements_par_menage',
    'commune_taux_etablissements_10_plus',
    'commune_sante_score_2013_commune',
    'commune_education_score_2013_commune',
    'commune_revenu_score_2013_commune',
    'commune_idh2_2013_commune',
    'commune_taux_criminalite_moyen'
]

print(f"Shape avant merge statique : {raw_idf_data.shape}")


raw_idf_data = raw_idf_data.merge(
    data_par_commune[colonnes_statiques],
    on='code_commune',
    how='left'
)

print(f"Shape après merge statique : {raw_idf_data.shape}")

Shape avant merge statique : (634236, 30)
Shape après merge statique : (634236, 55)


In [57]:
# Mapping année de référence -> taux de croissance à utiliser
# Pour une vente en 2021 , on prend le taux 2016-2020
mapping_taux_croissance = {
    2020: 'commune_taux_croissance_pop_annuel_2016_2019',
    2021: 'commune_taux_croissance_pop_annuel_2016_2020',
    2022: 'commune_taux_croissance_pop_annuel_2016_2021',
    2023: 'commune_taux_croissance_pop_annuel_2016_2022',
    2024: 'commune_taux_croissance_pop_annuel_2016_2023', 
    2025: 'commune_taux_croissance_pop_annuel_2016_2024'
}

# Mapping année de référence -> densité de population à utiliser
# Pour une vente en 2021, on prend la densité 2020
mapping_densite = {
    2020: 'commune_densite_pop_2019',
    2021: 'commune_densite_pop_2020',
    2022: 'commune_densite_pop_2021',
    2023: 'commune_densite_pop_2022',
    2024: 'commune_densite_pop_2023', 
    2025: 'commune_densite_pop_2024'
}

# On va créer un mapping code_commune + année_reference -> valeurs
# Préparer la table de lookup pour les taux de croissance et densités
taux_croissance_cols = list(mapping_taux_croissance.values())
densite_cols = list(mapping_densite.values())

# Créer une table longue avec code_commune et année_reference
lookup_data = []

for code_commune in data_par_commune['code_commune'].unique():
    commune_row = data_par_commune[data_par_commune['code_commune'] == code_commune].iloc[0]
    
    # Pour chaque année de référence possible
    for annee, col_taux in mapping_taux_croissance.items():
        for annee_dens, col_dens in mapping_densite.items():
            if annee == annee_dens:  # Même année de référence
                lookup_data.append({
                    'code_commune': code_commune,
                    'annee_vente': annee,
                    'commune_taux_croissance_pop': commune_row.get(col_taux, np.nan),
                    'commune_densite_pop': commune_row.get(col_dens, np.nan)
                })

lookup_table = pd.DataFrame(lookup_data)
print(f"Table de lookup créée : {lookup_table.shape}")
lookup_table.head(10)

Table de lookup créée : (7722, 4)


,code_commune,annee_vente,commune_taux_croissance_pop,commune_densite_pop
0,78638,2020,0.752379,574.792899
1,78638,2021,0.924149,583.076923
2,78638,2022,0.746816,583.313609
3,78638,2023,0.895685,592.899408
4,78638,2024,0.795949,594.082840
5,78638,2025,0.853214,601.538462
6,77295,2020,1.425548,157.224771
7,77295,2021,1.232726,158.256881
8,77295,2022,1.028839,158.600917
9,77295,2023,0.905193,159.059633


In [58]:
raw_idf_data = raw_idf_data.merge(
    lookup_table,
    left_on=['code_commune', 'annee'],
    right_on=['code_commune', 'annee_vente'],
    how='left'
).drop(columns=['annee_vente'])


print(f"Shape finale après merge temporel : {raw_idf_data.shape}")

Shape finale après merge temporel : (634236, 57)


In [59]:
raw_idf_data.isnull().mean() * 100

date_mutation                                       0.000000
nature_mutation                                     0.000000
valeur_fonciere                                     0.000000
adresse_numero                                      0.237451
adresse_nom_voie                                    0.000158
adresse_code_voie                                   0.000000
code_postal                                         0.000473
code_commune                                        0.000000
nom_commune                                         0.000000
code_departement                                    0.000000
id_parcelle                                         0.000000
nombre_lots                                         0.000000
code_type_local                                     0.000000
type_local                                          0.000000
surface_reelle_bati                                 0.000000
nombre_pieces_principales                           0.000000
surface_terrain         

# 4. Ajout des séries macro et des POIs

Règles générales : 

- quand on dispose de données mensuelles (ex: 2025-10), on va modifier la date au dernier jour d'Octobre 2025, afin de pouvoir joindre avec note base, en fonction des dates (format /AAAA/MM/JJ),

- quand on dispose de données trimestrielles (ex: 2025-T3), on remplace cette date par la date de fin de trimestre, ici : 2025-09-30

Pourquoi ? Parce que l'on va joindre et réaliser des lag, comme ça on se retrouve avec des intervalles pour selectioner les bonnes valeurs en fonction.

Evolution de l'indice des prix à la consommation (IPC),

Source : INSEE, https://www.insee.fr/fr/statistiques/8667361

In [60]:
df_evolution_ipc = pd.read_csv("data/raw/series_macro/indice_prix_conso.csv", sep=";")

In [61]:
df_evolution_ipc.head(10)

,Libellé,Indice des prix à la consommation - Base 2015 - Ensemble des ménages - France - Alimentation,Codes
0,idBank,001759963,NaN
1,Dernière mise à jour,31/10/2025 08:45,NaN
2,Période,NaN,NaN
3,2025-10,133.48,P
4,2025-09,133.79,A
5,2025-08,134.05,A
6,2025-07,133.64,A
7,2025-06,133.59,A
8,2025-05,133.75,A
9,2025-04,133.14,A


In [62]:
df_evolution_ipc = df_evolution_ipc.drop([0, 1, 2]).reset_index(drop=True)

df_evolution_ipc = df_evolution_ipc.rename(columns={
    df_evolution_ipc.columns[0]: "Date_maj",
    df_evolution_ipc.columns[1]: "IPC_base_2015"
})

df_evolution_ipc["Date_maj"] = pd.to_datetime(df_evolution_ipc["Date_maj"], format="%Y-%m", errors="coerce")

df_evolution_ipc["Date_maj"] = df_evolution_ipc["Date_maj"] + pd.offsets.MonthEnd(0)

df_evolution_ipc = df_evolution_ipc[df_evolution_ipc["Date_maj"] < "2025-07"]

df_evolution_ipc = df_evolution_ipc.drop(columns=[df_evolution_ipc.columns[-1]])

df_evolution_ipc.head(10)

,Date_maj,IPC_base_2015
4,2025-06-30,133.59
5,2025-05-31,133.75
6,2025-04-30,133.14
7,2025-03-31,132.19
8,2025-02-28,131.84
9,2025-01-31,131.94
10,2024-12-31,131.54
11,2024-11-30,131.72
12,2024-10-31,131.78
13,2024-09-30,131.52


Evolution de l'indice de confiance des ménages

Source : INSEE, https://www.insee.fr/fr/statistiques/8645671

In [63]:
df_indice_confiance_menage = pd.read_excel("data/raw/series_macro/indice_confiance_menage.xlsx")
df_indice_confiance_menage.head(10)

,Opinion des ménages-Monthly confidence consumer survey,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,Données corrigées des variations saisonnières-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DATE,Indicateur synthétique,Niveau de vie en France - évolution passée,Niveau de vie en France - perspectives d'évolu...,Chômage - perspectives d'évolution,Prix - évolution passée,Prix - perspectives d'évolution,Opportunité de faire des achats importants,Opportunité d'épargner,Capacité d'épargne actuelle,Situation financière personnelle - évolution p...,Situation financière personnelle - perspective...,Capacité d'épargne future
3,NaN,Synthetic index,"General economic situation, past 12 months","General economic situation, next 12 months","Unemployment, next 12 months","Consumer prices, past 12 months","Consumer prices, next 12 months","Major purchases intentions, next 12 months","Savings intentions, next 12 months",Current saving capacity,"Financial situation, past 12 months","Financial situation, next 12 months",Expected saving capacity
4,moyenne/average,100,-48.372204,-28.212314,32.515205,-12.451166,-31.539398,-15.636293,18.520498,9.824172,-21.064311,-6.665259,-6.661518
5,1972-10-01 00:00:00,127.552838,17.43,11.98,1.85,67.64,-26.9,29.35,-14.28,13.01,-3.4,12.42,-9.46
6,1973-01-01 00:00:00,130.040155,22.46,18.85,-1.42,57.63,-43.93,22.49,-5.01,13.87,0.36,14.32,-10.02
7,1973-05-01 00:00:00,131.094251,29.03,15.73,8.82,45.69,-16.11,29.12,-10.37,15.5,0.16,15.52,-11.79
8,1973-10-01 00:00:00,125.091056,12.13,3.64,1.63,79.49,-18.1,30.46,-21.78,11.82,-4.96,11.71,-14.51
9,1974-01-01 00:00:00,112.615392,2.47,-31.13,47.21,82.26,12.77,34.79,-30.09,10.03,-9.45,-6.85,-23.77


In [64]:
df_indice_confiance_menage = df_indice_confiance_menage.iloc[:, [0, 1]]
df_indice_confiance_menage = df_indice_confiance_menage.drop([0,1,2,3,4,5,6]).reset_index(drop=True)

df_indice_confiance_menage = df_indice_confiance_menage.rename(columns={
    df_indice_confiance_menage.columns[0]: "Date_maj",
    df_indice_confiance_menage.columns[1]: "Indice_confiance_menage"
})

df_indice_confiance_menage["Date_maj"] = pd.to_datetime(df_indice_confiance_menage["Date_maj"], format="%Y-%m", errors="coerce")
df_indice_confiance_menage["Date_maj"] = df_indice_confiance_menage["Date_maj"] + pd.offsets.MonthEnd(0)
df_indice_confiance_menage = df_indice_confiance_menage.sort_values(by="Date_maj", ascending=False)
df_indice_confiance_menage

,Date_maj,Indice_confiance_menage
505,2025-09-30,87.380365
504,2025-08-31,86.991779
503,2025-07-31,88.310865
502,2025-06-30,88.244595
501,2025-05-31,88.158058
...,...,...
4,1974-10-31,108.117889
3,1974-05-31,117.12321
2,1974-01-31,112.615392
1,1973-10-31,125.091056


In [65]:
df_indice_confiance_menage = df_indice_confiance_menage[
    (df_indice_confiance_menage["Date_maj"] >= "2019-01-01") &
    (df_indice_confiance_menage["Date_maj"] < "2025-07-01")  
]

Evolution du PIB en volume (%), base 2020

Source : Insee, https://www.insee.fr/fr/statistiques/2830547

In [66]:
df_evolution_pib_volume_base2020 = pd.read_excel("data/raw/series_macro/evolution_pib_volume_base_2020.xlsx")
df_evolution_pib_volume_base2020.head(10)

,Évolution du produit intérieur brut et de ses composantes,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,par rapport au trimestre précédent en volume en %,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Trimestre,Produit intérieur brut (PIB),Importations,Dépense de consommation des ménages,Dépense de consommation des APU1,Formation brute de capital fixe,dont :,NaN,NaN,Exportations,Contributions :,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,Entreprises non financières,ménages,APU1,NaN,Demande intérieure finale hors stocks,Variations de stocks,Commerce extérieur
4,2025-T3,0.504328,-0.363636,0.094047,0.5,0.399899,0.880356,-0.393351,-0.043548,2.230915,0.260384,-0.631701,0.875645
5,2025-T2,0.333917,1.388653,0.072989,0.5,0.007825,0.17428,-0.069843,-0.335146,0.315013,0.160025,0.547812,-0.37392
6,2025-T1,0.088369,0.14764,-0.287024,0.2,-0.156824,0.052357,-0.03495,-0.868242,-1.385435,-0.114786,0.721858,-0.518703
7,2024-T4,-0.047944,0.816698,0.03815,0.4,0.113526,0.095198,0.776311,-0.64205,1.471592,0.155161,-0.423387,0.220282
8,2024-T3,0.339255,0.530916,0.853694,0.3,-0.900275,-1.543234,-0.409946,-0.06485,-1.641456,0.332873,0.772414,-0.766033
9,2024-T2,0.1913,0.495462,0.039635,0.2,0.277694,0.046154,-1.052832,1.869435,1.535273,0.139623,-0.308459,0.360136


In [67]:
df_evolution_pib_volume_base2020 = df_evolution_pib_volume_base2020.rename(columns={
    df_evolution_pib_volume_base2020.columns[0]: "Date",
    df_evolution_pib_volume_base2020.columns[1]: "Evolution_PIB_volume_en_%_base2020"
})

df_evolution_pib_volume_base2020 = df_evolution_pib_volume_base2020.iloc[:, [0, 1]]

df_evolution_pib_volume_base2020.head(10)

,Date,Evolution_PIB_volume_en_%_base2020
0,NaN,NaN
1,par rapport au trimestre précédent en volume en %,NaN
2,Trimestre,Produit intérieur brut (PIB)
3,NaN,NaN
4,2025-T3,0.504328
5,2025-T2,0.333917
6,2025-T1,0.088369
7,2024-T4,-0.047944
8,2024-T3,0.339255
9,2024-T2,0.1913


In [68]:
df_evolution_pib_volume_base2020 = df_evolution_pib_volume_base2020.drop([0,1,2,3]).reset_index(drop=True)
df_evolution_pib_volume_base2020

,Date,Evolution_PIB_volume_en_%_base2020
0,2025-T3,0.504328
1,2025-T2,0.333917
2,2025-T1,0.088369
3,2024-T4,-0.047944
4,2024-T3,0.339255
...,...,...
306,1. Administrations publiques.,NaN
307,Note : données révisées ; les volumes sont mes...,NaN
308,"Lecture : au 3e trimestre 2025, le produit int...",NaN
309,Champ : France.,NaN


In [69]:
df_evolution_pib_volume_base2020['Date_Modif'] = df_evolution_pib_volume_base2020['Date'].apply(trimestre_to_date_fin)

# Vérifier les valeurs qui n'ont pas pu être converties
valeurs_problematiques = df_evolution_pib_volume_base2020[df_evolution_pib_volume_base2020['Date_Modif'].isna()]['Date'].unique()
print("Valeurs non converties:", valeurs_problematiques)

# Aperçu des résultats
print("\nAperçu des conversions:")
df_evolution_pib_volume_base2020[['Date', 'Date_Modif']].head(10)

Valeurs non converties: ['1. Administrations publiques.'
 "Note : données révisées\xa0; les volumes sont mesurés aux prix de l'année précédente chaînés et corrigés des variations saisonnières et des effets des jours ouvrables."
 'Lecture\xa0: au 3e trimestre 2025, le produit intérieur brut (PIB) en volume augmente de 0,5 % par rapport au trimestre précédent.'
 'Champ\xa0: France.'
 'Source : Insee, comptes nationaux trimestriels - base 2020.']

Aperçu des conversions:


,Date,Date_Modif
0,2025-T3,2025-09-30
1,2025-T2,2025-06-30
2,2025-T1,2025-03-31
3,2024-T4,2024-12-31
4,2024-T3,2024-09-30
5,2024-T2,2024-06-30
6,2024-T1,2024-03-31
7,2023-T4,2023-12-31
8,2023-T3,2023-09-30
9,2023-T2,2023-06-30


In [70]:
df_evolution_pib_volume_base2020 = df_evolution_pib_volume_base2020[df_evolution_pib_volume_base2020["Date_Modif"]>="2019-01-01"]

In [71]:
df_evolution_pib_volume_base2020.head()

,Date,Evolution_PIB_volume_en_%_base2020,Date_Modif
0,2025-T3,0.504328,2025-09-30
1,2025-T2,0.333917,2025-06-30
2,2025-T1,0.088369,2025-03-31
3,2024-T4,-0.047944,2024-12-31
4,2024-T3,0.339255,2024-09-30


Importation des données relatives aux taux moyen des crédits du secteur concurrentiel (hors assurance et coût des sûretés),

Source : Observatoire Crédit Logement CSA

In [72]:
df_taux_credit_moyen = pd.read_excel("data/raw/series_macro/Taux_moyen_credit_bancaire_immobilier_particulier_trimestriel.xlsx")

In [73]:
df_taux_credit_moyen.head(10)

,Date,Taux des prêts du secteur concurrentiel (hors assurance et coût des sureteé) en moyenne
0,2019-T1,0.0142
1,2019-T2,0.0129
2,2019-T3,0.0119
3,2019-T4,0.0113
4,2020-T1,0.0113
5,2020-T2,0.0129
6,2020-T3,0.0123
7,2020-T4,0.0120
8,2021-T1,0.0113
9,2021-T2,0.0106


In [74]:
df_taux_credit_moyen = df_taux_credit_moyen.rename(columns={
    df_taux_credit_moyen.columns[0]: "Date_Trimestriel",
    df_taux_credit_moyen.columns[1]: "Taux_moyen_credit_immo"
})

df_taux_credit_moyen['Date_Modif'] = df_taux_credit_moyen['Date_Trimestriel'].apply(trimestre_to_date_fin)

In [75]:
df_taux_credit_moyen.head(10)

,Date_Trimestriel,Taux_moyen_credit_immo,Date_Modif
0,2019-T1,0.0142,2019-03-31
1,2019-T2,0.0129,2019-06-30
2,2019-T3,0.0119,2019-09-30
3,2019-T4,0.0113,2019-12-31
4,2020-T1,0.0113,2020-03-31
5,2020-T2,0.0129,2020-06-30
6,2020-T3,0.0123,2020-09-30
7,2020-T4,0.0120,2020-12-31
8,2021-T1,0.0113,2021-03-31
9,2021-T2,0.0106,2021-06-30


In [76]:
print(df_taux_credit_moyen.columns)
print(df_evolution_pib_volume_base2020.columns)
print(df_indice_confiance_menage.columns)
print(df_evolution_ipc.columns)

Index(['Date_Trimestriel', 'Taux_moyen_credit_immo', 'Date_Modif'], dtype='object')
Index(['Date', 'Evolution_PIB_volume_en_%_base2020', 'Date_Modif'], dtype='object')
Index(['Date_maj', 'Indice_confiance_menage'], dtype='object')
Index(['Date_maj', 'IPC_base_2015'], dtype='object')


Création des lags mensuels/trimestrils

In [77]:
df_step1 = creer_lags(
    df_base=raw_idf_data,
    df_macro=df_taux_credit_moyen,
    col_date_base='date_mutation',
    col_date_macro='Date_Modif',
    colonnes_features=['Taux_moyen_credit_immo'],
    lags=[1, 2, 3, 4],
    frequence = "trimestriel"
)


→ Création lag1t (trimestriel)... ✓
→ Création lag2t (trimestriel)... ✓
→ Création lag3t (trimestriel)... ✓
→ Création lag4t (trimestriel)... ✓


In [78]:
df_step2 = creer_lags(
    df_base=df_step1,
    df_macro=df_evolution_pib_volume_base2020,
    col_date_base='date_mutation',
    col_date_macro='Date_Modif',
    colonnes_features=['Evolution_PIB_volume_en_%_base2020'],
    lags=[1, 2, 3, 4],
    frequence = "trimestriel"
)

→ Création lag1t (trimestriel)... ✓
→ Création lag2t (trimestriel)... ✓
→ Création lag3t (trimestriel)... ✓
→ Création lag4t (trimestriel)... ✓


In [79]:
df_step3 = creer_lags(
    df_base=df_step2,
    df_macro=df_indice_confiance_menage,
    col_date_base='date_mutation',
    col_date_macro='Date_maj',
    colonnes_features=['Indice_confiance_menage'],
    lags=[1, 2, 3, 6],
    frequence = "mensuel"
)

→ Création lag1m (mensuel)... ✓
→ Création lag2m (mensuel)... ✓
→ Création lag3m (mensuel)... ✓
→ Création lag6m (mensuel)... ✓


In [80]:
df_step4 = creer_lags(
    df_base=df_step3,
    df_macro=df_evolution_ipc,
    col_date_base='date_mutation',
    col_date_macro='Date_maj',
    colonnes_features=['IPC_base_2015'],
    lags=[1, 3, 6, 12],
    frequence = "mensuel"
)

→ Création lag1m (mensuel)... ✓
→ Création lag3m (mensuel)... ✓
→ Création lag6m (mensuel)... ✓
→ Création lag12m (mensuel)... ✓


In [81]:
raw_idf_data = df_step4.copy()

In [82]:
df_step4.shape

(634236, 73)

In [83]:
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,Evolution_PIB_volume_en_%_base2020_lag3t,Evolution_PIB_volume_en_%_base2020_lag4t,Indice_confiance_menage_lag1m,Indice_confiance_menage_lag2m,Indice_confiance_menage_lag3m,Indice_confiance_menage_lag6m,IPC_base_2015_lag1m,IPC_base_2015_lag3m,IPC_base_2015_lag6m,IPC_base_2015_lag12m
0,2021-01-01,Vente,169500.0,28.0,ALL HOCHE,4440,92130.0,92040,Issy-les-Moulineaux,92,...,-12.214629,-5.068832,96.156334,88.932449,93.925217,93.339790,107.989998,107.739998,107.849998,107.320000
1,2021-01-02,Vente,185000.0,20.0,AV GAL LECLERC,0360,95250.0,95051,Beauchamp,95,...,-12.214629,-5.068832,96.156334,88.932449,93.925217,93.339790,107.989998,107.739998,107.849998,107.320000
2,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,-12.214629,-5.068832,96.156334,88.932449,93.925217,93.339790,107.989998,107.739998,107.849998,107.320000
3,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,-12.214629,-5.068832,96.156334,88.932449,93.925217,93.339790,107.989998,107.739998,107.849998,107.320000
4,2021-01-04,Vente,255000.0,5.0,CHE DU MARCREUX,6115,93300.0,93001,Aubervilliers,93,...,-12.214629,-5.068832,96.156334,88.932449,93.925217,93.339790,107.989998,107.739998,107.849998,107.320000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634231,2025-06-30,Vente,283000.0,4.0,RUE FRANCOIS MITTERRAND,0347,77380.0,77122,Combs-la-Ville,77,...,0.339255,0.191300,88.158058,91.212196,91.188835,88.371445,133.750000,132.190002,131.539993,131.789993
634232,2025-06-30,Vente,144434.0,1.0,RUE JEAN DUSSART,1450,91390.0,91434,Morsang-sur-Orge,91,...,0.339255,0.191300,88.158058,91.212196,91.188835,88.371445,133.750000,132.190002,131.539993,131.789993
634233,2025-06-30,Vente,300000.0,32.0,RUE DU VAL ANDRE,0150,78560.0,78502,Le Port-Marly,78,...,0.339255,0.191300,88.158058,91.212196,91.188835,88.371445,133.750000,132.190002,131.539993,131.789993
634234,2025-06-30,Vente,476000.0,1.0,IMP DE LA FORET,0322,78450.0,78674,Villepreux,78,...,0.339255,0.191300,88.158058,91.212196,91.188835,88.371445,133.750000,132.190002,131.539993,131.789993


Ajouts des points d'intérêts dans un rayon de 500m 1000m et 2000m


- CATEGORIES_POI = {
    - 'education': ['school', 'kindergarten', 'college', 'university', 'library'],
    'commerce': ['supermarket', 'convenience', 'mall', 'department_store', 'bakery', 'butcher', 'greengrocer', 'marketplace'],
    
    - 'restauration': ['restaurant', 'cafe', 'fast_food', 'bar', 'pub', 'food_court', 'ice_cream', 'biergarten'],
    'sante': ['hospital', 'clinic', 'doctors', 'dentist', 'pharmacy', 'veterinary', 'nursing_home'],
    
    - 'loisirs': ['cinema', 'theatre', 'arts_centre', 'nightclub', 'casino', 'community_centre', 'social_facility', 'park', 'playground', 'sports_centre', 'swimming_pool', 'pitch', 'stadium'],
    
    - 'services': ['bank', 'atm', 'post_office', 'post_box', 'police', 'fire_station', 'townhall', 'courthouse', 'embassy']
}

- CATEGORIES_TRANSPORT = {
    - 'transport_lourd': ['railway_station', 'station', 'halt', 'tram_stop'],
    
    - 'bus': ['bus_stop', 'bus_station'],
    
    - 'transport_autre': ['taxi', 'ferry_terminal', 'aerodrome']
}

Chargement des POI en IDF pour les années 2020 à 2025

In [84]:
preparer_toutes_annees([2020, 2021, 2022, 2023, 2024, 2025])

TÉLÉCHARGEMENT & EXTRACTION - DONNÉES GÉOFABRIK

2020 déjà téléchargé
2021 déjà téléchargé
2022 déjà téléchargé
2023 déjà téléchargé
2024 déjà téléchargé
2025 déjà téléchargé

Toutes les données sont prêtes !


### Limite de l’étude pour les points d’intérêts “POI” (à prendre en compte lors du rapport): 

La plateforme Geofabrik, source des données OpenStreetMap, ne conserve les extraits mensuels que pour les 12 derniers mois. Au-delà de cette période, seuls les instantanés annuels du 1er janvier restent disponibles. Notre étude utilise donc les données du 1er janvier pour les années 2020 à 2025. 

Informations à avoir en tête, dans les données OpenStreetMap, on peut faire face à 3 types de géométries pour représenter un lieu lorsque l’on importe les données :

-   un point : = une coordonnée (x,y), ex: la localisation latitude longitude d’un bien immobilier, 
-   un polygone :  surface fermée définie par plusieurs points reliés (ex : un parc, un centre commercial)
-   un centroïde : centre géométrique d’un polygone (calculé une fois que l’on a converti les coordonnées)


On passe par les centroïdes pour calculer la distance géométrique d’un bien immobilier aux différents POI.


Ainsi, comme on souhaite déterminer le nombre de POI à proximité d’une localisation particulière, il nous faut projeter les coordonnées géographiques (latitude, longitude) dans un système de coordonnées projetées (en mètres), afin de pouvoir calculer des distances euclidiennes.

La projection officielle en France est le “Lambert 93”, cette projection (conique lié au système géodésique RGF93) est conçue pour minimiser les distorsions sur le territoire français (métropolitain), et est utilisée par les administrations françaises.

Exemple de conversion :

Une coordonnée WGS84 (ex: 48.8566° N, 2.3522°E) devient en Lambert 93 : 
-   X = 652785 m
-   Y = 6862000 m


In [85]:
# Ajout de chaque POI selon distance 
for rayon in [500, 1000, 2000]:
    raw_idf_data = traiter_dataset_complet(raw_idf_data, rayon=rayon, batch_size=100000)

raw_idf_data


PIPELINE COMPLET - ENRICHISSEMENT GÉOGRAPHIQUE

Années détectées: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Total: 634,236 transactions

ANNÉE 2021 - 172,742 transactions
Chargement des données Géofabrik 2021...
2021 déjà téléchargé
   ✓ 52,825 POIs | 35,361 transports

 Traitement par batches de 100,000 lignes...

 Traitement 2021_batch1 (100,000 transactions, rayon=500m)
 100,000 coordonnées valides (100.0%)
   → POIs... ✓
   → Transports... ✓
   ✓ Total moyen: 32.8 POIs

 Traitement 2021_batch2 (72,742 transactions, rayon=500m)
 72,742 coordonnées valides (100.0%)
   → POIs... ✓
   → Transports... ✓
   ✓ Total moyen: 32.5 POIs

ANNÉE 2022 - 168,734 transactions
Chargement des données Géofabrik 2022...
2022 déjà téléchargé
   ✓ 57,876 POIs | 35,569 transports

 Traitement par batches de 100,000 lignes...

 Traitement 2022_batch1 (100,000 transactions, rayon=500m)
 100,000 coordonnées valides (100.0%)
   → POIs... ✓
   → Transports... ✓
   ✓ Tot

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,nb_restauration_2000m,nb_sante_2000m,nb_loisirs_2000m,nb_services_2000m,nb_transport_lourd_2000m,nb_bus_2000m,nb_transport_autre_2000m,nb_total_2000m,diversite_2000m,densite_rel_2000m
0,2021-01-01,Vente,169500.0,28.0,ALL HOCHE,4440,92130.0,92040,Issy-les-Moulineaux,92,...,14,5,335,6,32,351,9,825,9,1.210341
1,2021-01-02,Vente,185000.0,20.0,AV GAL LECLERC,0360,95250.0,95051,Beauchamp,95,...,13,1,160,4,1,113,1,329,9,-0.372779
2,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,16,3,96,4,1,211,0,366,8,-0.254683
3,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,16,3,96,4,1,211,0,366,8,-0.254683
4,2021-01-04,Vente,255000.0,5.0,CHE DU MARCREUX,6115,93300.0,93001,Aubervilliers,93,...,5,6,251,5,19,217,2,557,9,0.354946
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634231,2025-06-30,Vente,283000.0,4.0,RUE FRANCOIS MITTERRAND,0347,77380.0,77122,Combs-la-Ville,77,...,4,2,94,2,0,63,0,170,7,-1.089169
634232,2025-06-30,Vente,144434.0,1.0,RUE JEAN DUSSART,1450,91390.0,91434,Morsang-sur-Orge,91,...,8,1,213,6,12,203,0,488,8,-0.129552
634233,2025-06-30,Vente,300000.0,32.0,RUE DU VAL ANDRE,0150,78560.0,78502,Le Port-Marly,78,...,2,2,326,3,3,190,2,600,9,0.208427
634234,2025-06-30,Vente,476000.0,1.0,IMP DE LA FORET,0322,78450.0,78674,Villepreux,78,...,1,1,64,2,1,86,1,181,9,-1.055975


In [86]:
# Sauvegarder le dataset final
output_path = "data/processed/idf_vf_full.parquet"
raw_idf_data.to_parquet(output_path, index=False)

print(f"✓ Dataset final sauvegardé : {output_path}")
print(f"  - Shape : {raw_idf_data.shape}")
print(f"  - Période : {raw_idf_data['date_mutation'].min()} à {raw_idf_data['date_mutation'].max()}")
print(f"  - Nombre de communes : {raw_idf_data['code_commune'].nunique()}")

✓ Dataset final sauvegardé : data/processed/idf_vf_full.parquet
  - Shape : (634236, 118)
  - Période : 2021-01-01 00:00:00 à 2025-06-30 00:00:00
  - Nombre de communes : 1283
